In [183]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [36]:
words = open('names.txt', 'r').read().splitlines()
print(words[:5])
print(len(words))

['emma', 'olivia', 'ava', 'isabella', 'sophia']
32033


In [640]:
# Choose n-gram context window
n = 4

# Build vocab of itos and stoi
chars = sorted(list(set(''.join(words))))

# Vocabulary and mappings between string and integer
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

vocab_size = len(stoi)
embedding_size = 10

"""
For each name, keep a sliding window context of the last n characters.
For the first n-1 characters, populate the empty places with '.'
E.g. for n = 4, the sliding context windows for "cedric" would be
. . . -> c
. . c -> e
. c e -> d
c e d -> r
e d r -> i
d r i -> c
""";
# Dataset of X -> Y mappings
X, Y = [], []
for word in words:
    context = [0] * (n-1)
    for char in word + '.':
        ix = stoi[char]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [641]:
len(X), len(Y)

(228146, 228146)

In [642]:
"""
Now we have the dataset we need to start training. Lets build our model.

We're building our model following Bengio et al. 2003 (A Neural Probabilistic Language Model)
where instead of words, we will have a vocab of 27 characters (including the special char '.').

We will choose a context window n for each word.
Each input x is a list of n characters that maps to an ouptut y which is the (n+1)-th character in the word.

Each character is one-hot encoded and multiplied by a 27 x 10 lookup table C to be embedded in a 10 imensional vector space.

We will pass our inputs of chosen context window through a MLP of 4 layers of 100 neurons.
Each layer will have a linear transformation wx + b, and non-linear activation function tanh.
With deeper layers of linear transformation, we will have increasing variance. 
This will lead to a greater saturation of the tails of tanh, which will kill the neuron.
We need to combat this by normalizing the distribution of activations before passing them to tanh.

There are a few ways to do this:

Kaiming initialization, whereby we construct the standard deviation of the sampling distribution that 
weights are drawn from initialization by multiplying by "empirical gain" and dividing it by the square root of fan-in (number of inputs)

The "empirical gain" counteracts for the lost variance due to the squashing of the nonlinearity
For our tanh, this constant happens to be 5/3 (due to some math that figured out how much tanh squashes).
For a nonllinearity like ReLU (which squashes all negatives), it would be sqrt(2) to compensate for all the lost negatives.

Since output variance = input variance * fan-in, our target weight variance is 1/fan-in to cancel the linear inflation due to fan-in.
To get our target weight standard deviation, we square root by the target weight variance, which is sqrt(1/fan-in) = 1/sqrt(fan-in).

So we multiply by 5/3 which is the empirical gain to add back the variance lost due to tanh and divide by
sqrt(fan-in) to counter variance inflation due to fan-in to give us our Kaiming initialization for this model.
Therefore, the standard deviation of the sampling distribution that we draw our weights from should be k_std = (5/3)/sqrt(fan-in)

This solves the weight variance problem at initialization. However, as traiing progresses, weights will change and alter the distribution.
We need a dynamic method to correct this shifting distribution. Following Ioffe et al. 2015 (Batch Normalization...), we can use the
batch normalization method to dynamically correct the shifting distribution at every step. 

For each batch of inputs, we can center the mean at 0 and normalize the standard deviation of the activations.
Each activation x_i is normalized by subtracting the mean and dividing by the square root of the variance
plus a small constant epsilon to prevent division by zero if the variance is zero: x_i_hat = (x_i - mean)/sqrt(variance + epsilon)

But this creates another problem. The mean and standard deviation is batch dependent.
During training, each input is evaluated in the context of its own batch for its mean and standard deviation.

However, during inference, whereby we evaluate a single input at a time, we do not have local context from other inputs.
If we just threw the input in and let it pass by the activation layers unnormalized, it would result in garbage since 
the model's parameters were tuned on normalized activations.

So we keep track of a global "running mean" and "running variance" that we update during training
which we can use to normalize single input inferences that work with the parameters of the model.

The purpose of normalizing the inputs is to let it start off with a "baseline" normal input to the nonlinearity
so it doesn't get stuck at the "tails" of the activation function. Over time, as the network learns,
it will update some scaling paramter gamma and shifting paramter beta, slowly shifting it to the nonlinear tails of the nonlinearity.
If we didn't normalize it first, the variance could instantly throw it at tails of the nonlinearity and never return. 
""";



In [643]:
# n-1 input chars * C embeddings -> flatten -> fan-in -> 100 neurons x 5 layers -> 100 fan-in -> 27 logits per char -> softmax -> sample nth char

# Initialize lookup table for 27 characters with 10 embeddings each
C = torch.randn(vocab_size, embedding_size)

# Lets initialize our first layer. Remember that fan-in should be n-1 * embeddings.
fan_in = (n-1) * embedding_size

# Each of the 100 neurons will accept a fan-in of 27, and add one bias.
W1 = (torch.randn(fan_in, 100) * (5.0/3.0) * (fan_in**-0.5)).float() # Implement Kaiming initialization, 5/3 to add variance back to tanh, and 1/sqrt(fan_in) to prevent scaling variance due to fan_in size.
B1 = torch.zeros(100).float()

# Final layer of 100 x 27 to create logits for each of our 27 possible characters
W2 = (torch.randn(100, vocab_size) * (100**-0.5)).float() # Since W2 goes to logits -> CrossEntropyLoss right away and not tanh, we don't need to multiply 5/3 to add back lost variance due to tanh.
B2 = torch.zeros(vocab_size).float()

# Learned gamma and beta scale and shift parameters for the nomalized batches, moving them towards the tails of the nonlinearity as the model learns
y_scale = torch.ones(30)
b_shift = torch.zeros(30)

# Keep track of running mean and variance to normalize inputs for inference later
batch_running_mean = torch.zeros(1, 30).float()
batch_running_variance = torch.ones(1, 30).float()

parameters = [C, W1, B1, W2, B2, y_scale, b_shift]

for p in parameters:
    p.requires_grad = True

In [644]:
# First, one-hot encode all characters.

Xenc = []

for x in X:

    xenc = []
    for c in x:
        cxenc = [0] * 27
        cxenc[c] = 1
        xenc.append(cxenc)

    Xenc.append(xenc)

Xenc = torch.tensor(Xenc).float()

Xenc.shape, Xenc.dtype

(torch.Size([228146, 3, 27]), torch.float32)

In [649]:
# Create a loop

# For loss
criterion = torch.nn.CrossEntropyLoss()

batch_size = 1000

# For batch norm
epsilon = 0.00005
momentum = 0.01

def train(loops):

    global batch_running_mean, batch_running_variance
    
    for loop in range(loops):
        # Forward pass
        batch = torch.randint(0, X.shape[0], (batch_size,)) # Chooses 32 random integers between 0 and our dataset size
        
        X_batch = C[X[batch]].view(-1, 30)
        Y_batch = Y[batch]

        # Apply batch normalization
        
    
        batch_mean = X_batch.mean(0, keepdim=True)
        batch_variance = X_batch.var(0, keepdim=True, unbiased=False)

        X_batch_norm = y_scale * (X_batch - batch_mean) / torch.sqrt(batch_variance + epsilon) + b_shift
        
        # Change the running scale and shift
        with torch.no_grad():
            batch_running_mean += (batch_mean - batch_running_mean) * momentum
            batch_running_variance += (batch_variance - batch_running_variance) * momentum

        # Now that batch norm has been applied, it can be passsed into nonlinearity without saturating the tails.
        forward = torch.tanh(X_batch_norm @ W1 + B1)
        logits = forward @ W2 + B2
    
        loss = criterion(logits, Y_batch)
        
        if loop % 100 == 0:
            print(loop, loss.item())
        
        # Backward pass
    
        loss.backward()
        
        for p in parameters:
            p.data += -0.01 * p.grad
            p.grad.zero_() # Zero the gradients to reset for next backward pass

In [650]:
train(10000)

0 2.152552843093872
100 2.173797369003296
200 2.135744571685791
300 2.109373092651367
400 2.137427568435669
500 2.15370512008667
600 2.1790060997009277
700 2.129746198654175
800 2.123183488845825
900 2.1204874515533447
1000 2.1873693466186523
1100 2.175584316253662
1200 2.146763801574707
1300 2.1755518913269043
1400 2.100038766860962
1500 2.097771644592285
1600 2.232595443725586
1700 2.202881336212158
1800 2.250791072845459
1900 2.203795909881592
2000 2.2022876739501953
2100 2.106106996536255
2200 2.1506078243255615
2300 2.1669580936431885
2400 2.112335205078125
2500 2.0926196575164795
2600 2.124612808227539
2700 2.150683879852295
2800 2.1685266494750977
2900 2.147101402282715
3000 2.1103034019470215
3100 2.1437559127807617
3200 2.163950204849243
3300 2.1905806064605713
3400 2.1754746437072754
3500 2.2191812992095947
3600 2.1500244140625
3700 2.131544351577759
3800 2.142066717147827
3900 2.227907419204712
4000 2.1765387058258057
4100 2.1959567070007324
4200 2.1901166439056396
4300 2.14

In [652]:
@torch.no_grad()
def sample(n_samples=10):
    for _ in range(n_samples):
        out = []
        context = [0] * (n - 1)

        while True:
            x = C[torch.tensor([context])]
            x = x.view(1, -1)

            x_norm = y_scale * (x - batch_running_mean) / torch.sqrt(batch_running_variance + epsilon) + b_shift

            h = torch.tanh(x_norm @ W1 + B1)
            logits = h @ W2 + B2
            probs = F.softmax(logits, dim=1)

            ix = torch.multinomial(probs, num_samples=1).item()
            context = context[1:] + [ix]
            out.append(ix)

            if ix == 0:
                break

        print(''.join(itos[i] for i in out))

sample(20)

syla.
alexia.
jero.
myrionna.
amethlian.
zadrikt.
cashir.
marayna.
frani.
hai.
khlzena.
ana.
bellemailynn.
juakfi.
deeshalla.
clynn.
adariels.
colt.
alyas.
kamsrowataxey.
